In [1]:
import json
import time
import requests
import pandas as pd
import datetime as dt

In [2]:
market_symbol = "btcusd"
url = f"https://www.bitstamp.net/api/v2/ohlc/{market_symbol}/"

STEP = 86400   # 1 day, in seconds
YEARS = 5
LIMIT = 1000   # Bitstamp's max candles per request

end_time = int(dt.datetime.now().timestamp())
start_time = end_time - YEARS * 365 * STEP

In [3]:
# Bitstamp caps each request at 1000 candles, so page through with `start`
# until we reach the present.
candles = {}
cursor = start_time

while cursor < end_time:
    params = {"step": STEP, "limit": LIMIT, "start": cursor}
    resp = requests.get(url, params=params).json()
    ohlc = resp.get("data", {}).get("ohlc", [])
    if not ohlc:
        break

    for row in ohlc:
        candles[row["timestamp"]] = row

    last_ts = int(ohlc[-1]["timestamp"])
    if last_ts <= cursor:
        break
    cursor = last_ts + STEP
    time.sleep(0.3)  # be polite to the API

data = pd.DataFrame(candles.values())
data = data.astype({"timestamp": "int64", "open": "float64", "high": "float64",
                     "low": "float64", "close": "float64", "volume": "float64"})
data = data.sort_values("timestamp").reset_index(drop=True)

print(f"Fetched {len(data)} daily candles: "
      f"{dt.datetime.fromtimestamp(data['timestamp'].iloc[0])} → "
      f"{dt.datetime.fromtimestamp(data['timestamp'].iloc[-1])}")
data.head()

Fetched 1826 daily candles: 2021-08-18 01:00:00 → 2026-08-17 01:00:00


,timestamp,open,high,low,close,volume
0,1629244800,44631.47,46041.62,44218.73,44721.13,2247.934158
1,1629331200,44734.16,47114.99,43935.54,46764.30,2776.237685
2,1629417600,46759.29,49436.10,46645.77,49356.00,2815.551595
3,1629504000,49332.31,49833.02,48300.00,48884.34,1768.241566
4,1629590400,48883.84,49540.01,48080.17,49301.86,985.765333


In [4]:
data.to_csv("tutorial.csv", index=False)
print("Saved to tutorial.csv")

Saved to tutorial.csv
